# CartSpark — Market Basket Analysis Recommender System

**CMPE 256 Hackathon Submission**

This notebook demonstrates the implementation of an item-based market basket recommendation system using co-occurrence analysis and lift metrics.

## Overview
- **Approach**: Item-based recommendations using association rules (NOT collaborative filtering)
- **Metrics**: Lift, Confidence, Support, Co-occurrence counts
- **Dataset**: Retail transaction data with security/surveillance products
- **Backend**: FastAPI with real-time recommendation API
- **Frontend**: Interactive web UI with dynamic cart-based recommendations

## 1. Setup and Imports

Install required packages and import libraries for data processing and visualization.

In [ ]:
# Install required packages (uncomment if running in Colab)
# !pip install pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import combinations
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
sns.set_style('whitegrid')

print("✓ Libraries imported successfully")

## 2. Load and Explore Dataset

Load the retail transaction data and understand its structure.

In [ ]:
# Load dataset
# For Colab: Upload the CSV file or mount Google Drive
# For local: Use the path to your CSV file

csv_path = "./CartSpark_Recommender/Retail/CMPE256_Hackathon_market_basket_analysis_Release.csv"

# If running in Colab, uncomment and upload file:
# from google.colab import files
# uploaded = files.upload()
# csv_path = list(uploaded.keys())[0]

df = pd.read_csv(csv_path)

print(f"Dataset shape: {df.shape}")
print(f"Total transactions: {len(df)}")
print(f"\nFirst few rows:")
df.head()

## 3. Data Preprocessing

Transform the wide-format transaction data into a basket format suitable for market basket analysis.

In [ ]:
# Detect transaction and item columns
txn_col = 'transaction_id'
item_cols = [c for c in df.columns if c.startswith('item_')]

print(f"Transaction column: {txn_col}")
print(f"Item columns: {item_cols}")

# Melt the dataframe to long format
long_df = df.melt(
    id_vars=[txn_col], 
    value_vars=item_cols, 
    var_name='position', 
    value_name='item'
).dropna()

# Clean item names
long_df['item'] = long_df['item'].astype(str).str.strip()
long_df[txn_col] = long_df[txn_col].astype(str).str.strip()

print(f"\nLong format shape: {long_df.shape}")
print(f"Sample records:")
long_df.head(10)

In [ ]:
# Create baskets (list of items per transaction)
baskets = long_df.groupby(txn_col)['item'].apply(
    lambda s: sorted(set(s.tolist()))
).tolist()

print(f"Total baskets: {len(baskets)}")
print(f"\nSample baskets:")
for i, basket in enumerate(baskets[:3]):
    print(f"\nBasket {i+1} ({len(basket)} items):")
    for item in basket:
        print(f"  - {item}")

## 4. Compute Market Basket Metrics

Calculate item frequencies, pair co-occurrences, support, and lift values.

In [ ]:
# Count individual items and pairs
item_counter = Counter()
pair_counter = Counter()

def _pair(a: str, b: str) -> Tuple[str, str]:
    """Create a sorted tuple for consistent pair representation"""
    return tuple(sorted((a, b)))

for basket in baskets:
    # Count items
    item_counter.update(basket)
    # Count pairs
    for a, b in combinations(basket, 2):
        pair_counter[_pair(a, b)] += 1

total_transactions = len(baskets)

print(f"Unique items: {len(item_counter)}")
print(f"Unique pairs: {len(pair_counter)}")
print(f"Total transactions: {total_transactions}")

In [ ]:
# Calculate support for each item
item_support = {item: count / total_transactions for item, count in item_counter.items()}

# Display top items by support
top_items = sorted(item_support.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 items by support:")
for item, support in top_items:
    print(f"  {support*100:.2f}% - {item[:60]}")

## 5. Recommendation Algorithm

Implement the lift-based recommendation system that ranks candidates based on their co-occurrence with cart items.

In [ ]:
def calculate_lift_sum(cart: List[str], candidate: str) -> Tuple[float, int]:
    """
    Calculate the sum of lift values between a candidate item and all items in the cart.
    
    Returns:
        (lift_sum, co-occurrence_count_sum)
    """
    lift_sum = 0.0
    cooccurrence_sum = 0
    
    for cart_item in cart:
        pair_count = pair_counter.get(_pair(cart_item, candidate), 0)
        if pair_count == 0:
            continue
            
        # Calculate lift: P(A,B) / (P(A) * P(B))
        p_pair = pair_count / total_transactions
        p_cart_item = item_support[cart_item]
        p_candidate = item_support[candidate]
        
        lift = p_pair / (p_cart_item * p_candidate)
        lift_sum += lift
        cooccurrence_sum += pair_count
    
    return lift_sum, cooccurrence_sum

print("✓ Lift calculation function defined")

In [ ]:
def get_recommendations(cart: List[str], top_k: int = 8, min_pairs: int = 1, min_lift: float = 0.0):
    """
    Generate top-K recommendations for a given cart.
    
    Args:
        cart: List of items currently in the cart
        top_k: Number of recommendations to return
        min_pairs: Minimum co-occurrence count threshold
        min_lift: Minimum lift threshold
    
    Returns:
        List of recommended items with their metrics
    """
    # Filter cart to only include known items
    valid_cart = [item for item in cart if item in item_support]
    
    if not valid_cart:
        return []
    
    catalog = list(item_counter.keys())
    scored_items = []
    
    # Score each candidate item
    for candidate in catalog:
        if candidate in valid_cart:
            continue  # Don't recommend items already in cart
        
        lift_sum, cooccurrence_sum = calculate_lift_sum(valid_cart, candidate)
        
        # Apply thresholds
        if cooccurrence_sum >= min_pairs and lift_sum >= min_lift:
            scored_items.append({
                'item': candidate,
                'lift_sum': lift_sum,
                'cooccurrence_count': cooccurrence_sum,
                'support': item_support[candidate]
            })
    
    # Sort by lift_sum (primary) and cooccurrence_count (secondary)
    scored_items.sort(key=lambda x: (x['lift_sum'], x['cooccurrence_count']), reverse=True)
    
    return scored_items[:top_k]

print("✓ Recommendation function defined")

## 6. Test Recommendations with Sample Carts

Demonstrate how recommendations change as items are added to the cart.

In [ ]:
# Test Case 1: Single item in cart
test_cart_1 = ["Bosch B5512 Control Panel (SKU: B5512)"]

print("=" * 80)
print(f"CART: {test_cart_1[0]}")
print("=" * 80)

recommendations = get_recommendations(test_cart_1, top_k=5)

if recommendations:
    print(f"\nTop {len(recommendations)} Recommendations:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec['item'][:70]}")
        print(f"   Lift Sum: {rec['lift_sum']:.3f} | Co-occurrence: {rec['cooccurrence_count']} | Support: {rec['support']*100:.2f}%")
        print()
else:
    print("\nNo recommendations found.")

In [ ]:
# Test Case 2: Add second item to cart (dynamic re-ranking)
test_cart_2 = [
    "Bosch B5512 Control Panel (SKU: B5512)",
    "Hanwha QNV-6010R Network Camera (SKU: QNV-6010R)"
]

print("=" * 80)
print(f"CART (2 items):")
for item in test_cart_2:
    print(f"  - {item}")
print("=" * 80)

recommendations = get_recommendations(test_cart_2, top_k=5)

if recommendations:
    print(f"\nTop {len(recommendations)} Recommendations:\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec['item'][:70]}")
        print(f"   Lift Sum: {rec['lift_sum']:.3f} | Co-occurrence: {rec['cooccurrence_count']} | Support: {rec['support']*100:.2f}%")
        print()
else:
    print("\nNo recommendations found.")

In [ ]:
# Test Case 3: Larger cart (candidate set should prune)
test_cart_3 = [
    "Bosch B5512 Control Panel (SKU: B5512)",
    "Hanwha QNV-6010R Network Camera (SKU: QNV-6010R)",
    "DSC WS4916 Smoke Detector (SKU: WS4916)"
]

print("=" * 80)
print(f"CART (3 items):")
for item in test_cart_3:
    print(f"  - {item}")
print("=" * 80)

recommendations = get_recommendations(test_cart_3, top_k=5)

if recommendations:
    print(f"\nTop {len(recommendations)} Recommendations (candidate set pruned):\n")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec['item'][:70]}")
        print(f"   Lift Sum: {rec['lift_sum']:.3f} | Co-occurrence: {rec['cooccurrence_count']} | Support: {rec['support']*100:.2f}%")
        print()
else:
    print("\nNo recommendations found (graceful handling).")

## 7. Visualizations

Visualize the top item pairs and their lift values.

In [ ]:
# Calculate lift for top pairs
pair_metrics = []

for (item_a, item_b), count in pair_counter.most_common(20):
    p_pair = count / total_transactions
    p_a = item_support[item_a]
    p_b = item_support[item_b]
    lift = p_pair / (p_a * p_b)
    confidence_a_to_b = count / item_counter[item_a]
    confidence_b_to_a = count / item_counter[item_b]
    
    pair_metrics.append({
        'item_a': item_a[:30] + '...' if len(item_a) > 30 else item_a,
        'item_b': item_b[:30] + '...' if len(item_b) > 30 else item_b,
        'count': count,
        'lift': lift,
        'conf_a_b': confidence_a_to_b,
        'conf_b_a': confidence_b_to_a,
        'support': p_pair
    })

pair_df = pd.DataFrame(pair_metrics)
print("Top 20 Item Pairs by Co-occurrence:")
pair_df.head(10)

In [ ]:
# Visualize top pairs by lift
plt.figure(figsize=(12, 6))
top_10_pairs = pair_df.nlargest(10, 'lift')

pair_labels = [f"{row['item_a'][:20]}...\n+ {row['item_b'][:20]}..." 
               for _, row in top_10_pairs.iterrows()]

plt.barh(range(len(top_10_pairs)), top_10_pairs['lift'], color='steelblue')
plt.yticks(range(len(top_10_pairs)), pair_labels, fontsize=8)
plt.xlabel('Lift Value', fontsize=12)
plt.title('Top 10 Item Pairs by Lift', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nLift > 1.0 indicates positive correlation between items")

In [ ]:
# Distribution of basket sizes
basket_sizes = [len(basket) for basket in baskets]

plt.figure(figsize=(10, 5))
plt.hist(basket_sizes, bins=range(1, max(basket_sizes)+2), edgecolor='black', color='coral', alpha=0.7)
plt.xlabel('Basket Size (Number of Items)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Basket Sizes', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average basket size: {sum(basket_sizes)/len(basket_sizes):.2f} items")
print(f"Max basket size: {max(basket_sizes)} items")
print(f"Min basket size: {min(basket_sizes)} items")

## 8. Key Insights and Implementation Notes

### Algorithm Characteristics:
1. **Item-Based Approach**: Uses co-occurrence patterns, NOT collaborative filtering
2. **Dynamic Re-ranking**: Recommendations update with every cart change
3. **Basket-Aware**: Only unique items matter, quantities don't affect logic
4. **Flexible Thresholds**: Accepts low support (~5%) and confidence (~10%)
5. **Graceful Degradation**: Returns empty list when no patterns match
6. **Pruning**: Candidate set naturally shrinks as basket grows

### Metrics Used:
- **Lift**: Measures how much more likely items appear together vs. independently
- **Confidence**: Probability of item B given item A
- **Support**: Frequency of item/pair in all transactions
- **Co-occurrence Count**: Raw count of transactions containing both items

### Web Application:
- **Backend**: FastAPI serving `/api/recommend` endpoint
- **Frontend**: Real-time UI with cart, catalog, and recommendation panels
- **Features**: Search, quantity management, checkout flow, lift charts, explanations

## 9. Conclusion

This notebook demonstrates a complete market basket analysis recommendation system that:

✅ Generates item-based recommendations using association rules  
✅ Dynamically re-ranks suggestions based on current cart contents  
✅ Handles edge cases gracefully (no patterns → empty recommendations)  
✅ Uses flexible thresholds suitable for sparse retail data  
✅ Provides explainable recommendations with lift, confidence, and support metrics  

The full implementation includes:
- FastAPI backend (`backend/main.py`)
- Interactive web UI (`frontend/index.html`)
- Real-time recommendation updates
- Visual analytics with Chart.js

**Repository**: CartSpark_Recommender  
**Demo**: Run `run_api.bat` and `run_frontend.bat` to start the application